# MSigDB ORA Saturation Plot (study subsampling)

**Environment:** `clamp-analyses`

Reads per-model ORA caches produced by `00_bp_saturation_study_ora_analysis.ipynb` and plots MSigDB pathway coverage against K (number of LVs). Each line is one study coverage level; boxplots show seed distribution. Four plots: CLAMPfull and CLAMPbase at FDR 0.05 and FDR 0.01.

In [ ]:
library(here)
library(ggplot2)
library(dplyr)

input_dir  <- here("output/03_model_biology/00_archs4/01_pathway_coverage_bp_saturation_study/00_bp_saturation_study_ora_analysis")
output_dir <- here("output/03_model_biology/00_archs4/01_pathway_coverage_bp_saturation_study/01_bp_saturation_study_plot")
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

theme_ng <- function() {
  theme_classic(base_size = 18) +
    theme(
      axis.title        = element_text(size = 24, colour = "black"),
      axis.title.y      = element_text(margin = margin(r = 18)),
      axis.text         = element_text(size = 18, colour = "black"),
      axis.line         = element_line(linewidth = 0.8, colour = "black"),
      axis.ticks        = element_line(linewidth = 0.8, colour = "black"),
      axis.ticks.length = unit(0.22, "cm"),
      panel.grid.major.y = element_line(colour = "#BDBDBD", linewidth = 0.7),
      panel.grid.minor   = element_blank(),
      legend.position    = "right",
      strip.background   = element_blank(),
      strip.text         = element_text(face = "bold", size = 18),
      plot.margin        = margin(t = 16, r = 12, b = 12, l = 42)
    )
}

## Load results

In [ ]:
load_saturation_results <- function(model_subdir) {
  rds_files <- list.files(
    file.path(input_dir, model_subdir),
    pattern    = "^rs[0-9]+_k[0-9]+_seed[0-9]+_msigdb\\.rds$",
    full.names = TRUE
  )

  if (length(rds_files) == 0) {
    warning(sprintf("No _msigdb.rds files found for %s", model_subdir))
    return(NULL)
  }

  message("Loading ", length(rds_files), " caches from ", model_subdir)

  rows <- lapply(rds_files, function(f) {
    m <- regmatches(basename(f), regexec("^rs([0-9]+)_k([0-9]+)_seed([0-9]+)_msigdb\\.rds$", basename(f)))[[1]]
    if (length(m) < 4) return(NULL)
    res <- readRDS(f)
    if (is.null(res$terms_padj)) return(NULL)
    data.frame(
      rs_pct                = as.integer(m[2]),
      k_val                 = as.integer(m[3]),
      seed                  = as.integer(m[4]),
      n_studies             = if (!is.null(res$n_studies)) res$n_studies else NA_integer_,
      n_samples             = res$n_samples,
      n_lvs                 = res$n_lvs,
      n_top_genes           = res$n_top_genes,
      n_total_msigdb        = res$n_total_msigdb,
      coverage_msigdb_fdr05 = sum(res$terms_padj < 0.05) / res$n_total_msigdb,
      coverage_msigdb_fdr01 = sum(res$terms_padj < 0.01) / res$n_total_msigdb,
      stringsAsFactors      = FALSE
    )
  })

  df <- do.call(rbind, Filter(Negate(is.null), rows))
  df[order(df$rs_pct, df$k_val, df$seed), ]
}

results_full <- load_saturation_results("CLAMPfull")
results_base <- load_saturation_results("CLAMPbase")

k_values     <- sort(unique(results_full$k_val))
avail_levels <- paste0(sort(unique(results_full$rs_pct)), "%")

message("k_values: ",      paste(k_values,     collapse = ", "))
message("rs_pct levels: ", paste(avail_levels, collapse = ", "))

## Prepare plot data

In [ ]:
coverage_colors <- c(
  "1%"   = "#2166AC",
  "5%"   = "#4DAC26",
  "10%"  = "#D6604D",
  "25%"  = "#762A83",
  "50%"  = "#E08214",
  "75%"  = "#01665E",
  "100%" = "#000000"
)
coverage_shapes <- c("1%"=16, "5%"=3, "10%"=1, "25%"=17, "50%"=15, "75%"=8, "100%"=18)

make_plot_df <- function(df) {
  df %>%
    dplyr::mutate(
      data_label                = factor(paste0(rs_pct, "%"),
                                         levels = paste0(c(1, 5, 10, 25, 50, 75, 100), "%")),
      coverage_msigdb_fdr05_pct = coverage_msigdb_fdr05 * 100,
      coverage_msigdb_fdr01_pct = coverage_msigdb_fdr01 * 100
    )
}

make_line_df <- function(plot_df, y_col) {
  plot_df %>%
    dplyr::group_by(rs_pct, k_val, data_label) %>%
    dplyr::summarise(y = median(.data[[y_col]], na.rm = TRUE), .groups = "drop") %>%
    dplyr::arrange(data_label, k_val)
}

plot_full <- make_plot_df(results_full)
plot_base <- if (!is.null(results_base)) make_plot_df(results_base) else NULL

box_width <- diff(range(k_values)) * 0.04

## CLAMPfull: FDR 0.05

In [ ]:
make_sat_plot <- function(plot_df, y_col, y_label) {
  line_df <- make_line_df(plot_df, y_col)
  lvls    <- levels(plot_df$data_label)

  ggplot(plot_df, aes(x = k_val, y = .data[[y_col]],
                      colour = data_label, fill = data_label)) +
    geom_boxplot(
      aes(group = interaction(data_label, factor(k_val))),
      alpha = 0.2, outlier.shape = NA, width = box_width, position = "identity"
    ) +
    geom_line(data = line_df, aes(x = k_val, y = y, group = data_label), linewidth = 1.0) +
    geom_point(aes(shape = data_label), size = 2.5, stroke = 1.0) +
    scale_x_continuous(breaks = k_values) +
    scale_colour_manual(values = coverage_colors, name = "Study\ncoverage", limits = avail_levels) +
    scale_fill_manual(values   = coverage_colors, name = "Study\ncoverage", limits = avail_levels) +
    scale_shape_manual(values  = coverage_shapes, name = "Study\ncoverage", limits = avail_levels) +
    scale_y_continuous(
      labels = scales::percent_format(accuracy = 0.1, scale = 1),
      breaks = seq(0, 100, by = 5),
      limits = c(0, NA),
      expand = expansion(mult = c(0, 0.05))
    ) +
    labs(x = "K (number of LVs)", y = y_label) +
    theme_ng()
}

options(repr.plot.width = 10, repr.plot.height = 7)
p_full_05 <- make_sat_plot(plot_full, "coverage_msigdb_fdr05_pct", "MSigDB ORA coverage, FDR < 0.05 (%)")
p_full_05

In [ ]:
p_full_01 <- make_sat_plot(plot_full, "coverage_msigdb_fdr01_pct", "MSigDB ORA coverage, FDR < 0.01 (%)")
p_full_01

## CLAMPbase: FDR 0.05

In [ ]:
if (!is.null(plot_base)) {
  p_base_05 <- make_sat_plot(plot_base, "coverage_msigdb_fdr05_pct", "MSigDB ORA coverage, FDR < 0.05 (%)")
  p_base_05
} else {
  message("CLAMPbase results not available. Run 00_bp_saturation_study_ora_analysis.ipynb first.")
}

## CLAMPbase: FDR 0.01

In [ ]:
if (!is.null(plot_base)) {
  p_base_01 <- make_sat_plot(plot_base, "coverage_msigdb_fdr01_pct", "MSigDB ORA coverage, FDR < 0.01 (%)")
  p_base_01
} else {
  message("CLAMPbase results not available.")
}